# OOD Evaluation for Craigslist Models

This notebook loads a trained model and preprocessing metadata and evaluates it on:

- ID test split (from a chosen training run)
- OOD datasets constructed from numeric tails, rare manufacturers, and geographic shifts (FL+TX)

All cells start with a short comment describing their purpose.

In [ ]:
# Cell 1: imports, device, and core paths
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from data import DataConfig, _prepare_frame, _apply_encoders, inverse_target
from model_base import MLPRegressor

# Select device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Configure a specific training run to evaluate
# NOTE: update this path to point to a concrete run directory after training
run_dir = Path("outputs/trainings/base").resolve()
print("Current run_dir:", run_dir)

model_path = run_dir / "model.pt"
meta_path = run_dir / "preproc_meta.json"

# OOD datasets produced by scripts/make_ood_splits.py
datasets_dir = Path("datasets").resolve()
ood_files = {
    "tail_mpy_high": datasets_dir / "craigslist_ood_tail_mpy_high.csv",
    "tail_odo_high": datasets_dir / "craigslist_ood_tail_odo_high.csv",
    "tail_year_old": datasets_dir / "craigslist_ood_tail_year_old.csv",
    "tail_multi": datasets_dir / "craigslist_ood_tail_multi.csv",
    "rare_manufacturers": datasets_dir / "craigslist_ood_rare_manufacturers.csv",
    "geo_fl_tx": datasets_dir / "craigslist_ood_geo_fl_tx.csv",
}
print("OOD files:")
for k, v in ood_files.items():
    print(f"  {k}: {v}")


In [ ]:
# Cell 2: load preprocessing metadata and inspect encoder configuration
with open(meta_path, "r") as f:
    meta = json.load(f)

enc = meta["encoders"]
numeric_cols = meta["numeric_cols"]
onehot_cols = meta["onehot_cols"]
hash_cols = meta["hash_cols"]
hash_dims = meta["hash_dims"]
target_meta = meta.get("target", {})
feature_dim = int(meta["feature_dim"])

print("numeric_cols:", numeric_cols)
print("onehot_cols:", onehot_cols)
print("hash_cols:", hash_cols)
print("hash_dims:", hash_dims)
print("target_meta:", target_meta)
print("feature_dim:", feature_dim)


In [ ]:
# Cell 3: load training config and recreate the model architecture
import yaml

# Point to the training config used for this run
cfg_path = Path("configs/train_base.yaml").resolve()
with open(cfg_path, "r") as f:
    cfg = yaml.safe_load(f)

head_type = cfg["model"]["head_type"].lower()
hidden_dims = cfg["model"]["hidden_dims"]
activation = cfg["model"]["activation"]
dropout = cfg["model"]["dropout"]

model = MLPRegressor(
    in_dim=feature_dim,
    head_type=head_type,
    hidden_dims=hidden_dims,
    activation=activation,
    dropout=dropout,
).to(device)

ckpt = torch.load(model_path, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print("Loaded model from:", model_path)


In [ ]:
# Cell 4: helper to load and encode a dataset using training encoders
def load_encoded_dataset(csv_path: Path, target_col: str = "price"):
    """Load a CSV, apply the same _prepare_frame and encoders as in training."""
    csv_path = csv_path.resolve()
    cfg_local = DataConfig(csv_path=str(csv_path), target_col=target_col)
    df_prep = _prepare_frame(cfg_local)

    # Features
    X = _apply_encoders(df_prep, numeric_cols, onehot_cols, hash_cols, enc)
    X = X.astype(np.float32)

    # Targets (original space)
    y_orig = df_prep[target_col].astype(float).to_numpy()

    # Targets (transformed space) based on target_meta
    mode = target_meta.get("mode", "none").lower()
    if mode == "log1p":
        y_tr = np.log1p(y_orig)
    else:
        y_tr = y_orig.copy()

    return df_prep, X, y_tr, y_orig


In [ ]:
# Cell 5: helper to run the model and compute basic original-space metrics
def evaluate_dataset(X: np.ndarray, y_tr: np.ndarray, y_orig: np.ndarray, batch_size: int = 2048):
    """Run the model on X and compute MAE/RMSE in original price space."""
    X_tensor = torch.from_numpy(X)
    y_tr_tensor = torch.from_numpy(y_tr).view(-1, 1)

    preds_tr = []
    with torch.no_grad():
        for i in range(0, X_tensor.shape[0], batch_size):
            xb = X_tensor[i : i + batch_size].to(device)
            out = model(xb)
            mu = out["mu"]
            preds_tr.append(mu.cpu().numpy())

    mu_tr = np.concatenate(preds_tr, axis=0).reshape(-1, 1)

    # Inverse target transform to original price space
    mu_orig = inverse_target(mu_tr, target_meta).reshape(-1)

    # Original-space metrics
    ae = np.abs(mu_orig - y_orig)
    se = (mu_orig - y_orig) ** 2
    mae = float(np.mean(ae))
    rmse = float(np.sqrt(np.mean(se)))

    return {
        "n": int(len(y_orig)),
        "mae_orig": mae,
        "rmse_orig": rmse,
        "mu_orig": mu_orig,
    }


In [ ]:
# Cell 6: example evaluation on the geographic FL+TX OOD dataset
name = "geo_fl_tx"
csv_path = ood_files[name]

df_ood, X_ood, y_tr_ood, y_orig_ood = load_encoded_dataset(csv_path)
metrics_geo = evaluate_dataset(X_ood, y_tr_ood, y_orig_ood)

print(f"OOD dataset: {name}")
print({k: v for k, v in metrics_geo.items() if k != "mu_orig"})

# Inspect a few rows with predictions vs ground truth
sample = df_ood.copy()
sample["y_true"] = y_orig_ood
sample["y_pred"] = metrics_geo["mu_orig"]
display(sample.head(10))


In [ ]:
# Cell 7: loop over all OOD datasets and summarize MAE/RMSE
results = []
for name, csv_path in ood_files.items():
    df_ood, X_ood, y_tr_ood, y_orig_ood = load_encoded_dataset(csv_path)
    m = evaluate_dataset(X_ood, y_tr_ood, y_orig_ood)
    m["dataset"] = name
    results.append(m)

summary = pd.DataFrame(results)[["dataset", "n", "mae_orig", "rmse_orig"]]
display(summary)


In [ ]:
# Cell 8: optional – evaluate on ID test split using training splits from preproc_meta
splits = meta.get("splits", {})
test_indices = np.array(splits.get("test", []), dtype=int)

id_csv = datasets_dir / "craigslist_cleaned.csv"  # assumes ID-only file is in place
df_id, X_id, y_tr_id, y_orig_id = load_encoded_dataset(id_csv)

if test_indices.size > 0:
    X_id_te = X_id[test_indices]
    y_tr_id_te = y_tr_id[test_indices]
    y_orig_id_te = y_orig_id[test_indices]
else:
    X_id_te, y_tr_id_te, y_orig_id_te = X_id, y_tr_id, y_orig_id

metrics_id = evaluate_dataset(X_id_te, y_tr_id_te, y_orig_id_te)
print("ID test metrics:")
print({k: v for k, v in metrics_id.items() if k != "mu_orig"})
